# Comparación de modelos

Este notebook **no entrena nada**: solo carga las métricas exportadas por cada notebook de modelado y las concatena en una sola tabla.

Cada notebook fuente guarda su CSV en `Modelado/artifacts/`:

| Notebook fuente | Archivo de métricas |
|---|---|
| `JoinXcodex/Join.ipynb` (BLOQUE 10) | `metricas_baseline_sgd.csv` |
| `Modelado/modeladoRandomForest.ipynb` | `metricas_rf.csv` |
| `Modelado/modeladoHistGradientBoosting.ipynb` | `metricas_hgb.csv` |

Si falta alguno, ejecuta el notebook correspondiente hasta la celda de exportación de métricas.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ARTIFACTS_DIR = Path("artifacts")
print("Carpeta:", ARTIFACTS_DIR.resolve())
print("Archivos disponibles:")
for p in sorted(ARTIFACTS_DIR.glob("metricas_*.csv")):
    print("  -", p.name)

Carpeta: C:\Users\camil\OneDrive\Documentos\Aprendizaje de Maquina\Taller\Modelado\artifacts
Archivos disponibles:
  - metricas_baseline_sgd.csv
  - metricas_hgb.csv
  - metricas_rf.csv


In [2]:
archivos_metricas = {
    "baseline_sgd": ARTIFACTS_DIR / "metricas_baseline_sgd.csv",
    "random_forest": ARTIFACTS_DIR / "metricas_rf.csv",
    "hgb": ARTIFACTS_DIR / "metricas_hgb.csv",
}

dfs = []
for nombre, ruta in archivos_metricas.items():
    if ruta.exists():
        df = pd.read_csv(ruta)
        df["fuente"] = nombre
        dfs.append(df)
        print(f"[OK] {nombre}: {len(df)} fila(s)")
    else:
        print(f"[FALTA] {nombre} -> {ruta}")

if not dfs:
    raise FileNotFoundError(
        "No se encontraron CSV de métricas. Ejecuta antes la celda final de cada "
        "notebook de modelado para exportar los CSV."
    )

[OK] baseline_sgd: 2 fila(s)
[OK] random_forest: 1 fila(s)
[OK] hgb: 1 fila(s)


In [3]:
tabla_comparativa_modelos = pd.concat(dfs, ignore_index=True)

orden_cols = [
    "modelo", "umbral",
    "accuracy", "precision", "recall", "f1",
    "pr_auc", "roc_auc",
    "tp", "fp", "fn", "tn",
    "fuente",
]
orden_cols = [c for c in orden_cols if c in tabla_comparativa_modelos.columns]
tabla_comparativa_modelos = tabla_comparativa_modelos[orden_cols]

tabla_comparativa_modelos = tabla_comparativa_modelos.sort_values(
    "f1", ascending=False
).reset_index(drop=True)

display(tabla_comparativa_modelos.round(4))

,modelo,umbral,accuracy,precision,recall,f1,pr_auc,roc_auc,tp,fp,fn,tn,fuente
0,HistGradientBoosting balanceado,0.85,0.9583,0.1207,0.2353,0.1596,0.0834,0.8109,5770,42031,18747,1391864,hgb
1,Random Forest balanceado,0.8,0.9469,0.1042,0.2837,0.1524,0.0769,0.8069,6956,59825,17561,1374070,random_forest
2,SGD balanceado,0.8,0.9402,0.0928,0.2917,0.1409,0.0723,0.7768,7152,69880,17365,1364015,baseline_sgd
3,Baseline siempre 0,-,0.9832,0.0000,0.0000,0.0000,0.0168,NaN,0,0,24517,1433895,baseline_sgd


In [4]:
ruta_tabla = ARTIFACTS_DIR / "tabla_comparativa_modelos.csv"
tabla_comparativa_modelos.to_csv(ruta_tabla, index=False)
print(f"Tabla comparativa guardada en: {ruta_tabla.resolve()}")

Tabla comparativa guardada en: C:\Users\camil\OneDrive\Documentos\Aprendizaje de Maquina\Taller\Modelado\artifacts\tabla_comparativa_modelos.csv
